# Image-Constrained Generative Modeling of Retinal Vasculature

This notebook demonstrates the full pipeline:
1. Load a fundus image
2. Extract structural constraints (optic disc, macula, vessel orientation, density)
3. Generate vascular trees using both baseline and image-constrained models
4. Evaluate and compare the results

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cv2
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.constraint_extraction.extract_constraints import (
    extract_all_constraints,
    detect_retina_boundary,
    detect_optic_disc,
    segment_vessels,
)
from src.generative_models.branching_model import (
    TreeGeneratorConfig,
    RetinalTreeGenerator,
    ConstrainedTreeGenerator,
)
from src.evaluation.metrics import (
    coverage_score,
    coverage_uniformity,
    length_per_terminal,
    fractal_dimension_box_counting,
    branch_angle_statistics,
    density_correlation,
)
from src.visualization.plot_network import (
    plot_vascular_tree,
    plot_terminal_distribution,
    plot_constraints_overlay,
    plot_comparison,
    plot_density_map,
)

print("All modules loaded successfully.")

## 1. Load and Inspect a Fundus Image

In [ ]:
# Load a sample image
data_dir = project_root / "data" / "raw" / "healthy"
image_path = data_dir / "01_h.jpg"

image = cv2.imread(str(image_path))
print(f"Image shape: {image.shape}")
print(f"Image dtype: {image.dtype}")

plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title(f"Fundus Image: {image_path.name}")
plt.axis("off")
plt.show()

## 2. Extract Structural Constraints

In [ ]:
# Extract all constraints from the image
constraints = extract_all_constraints(image, density_grid_size=20)

print("=== Extracted Constraints ===")
print(f"Retina center: {constraints['retina_center']}, radius: {constraints['retina_radius']}")
print(f"Optic disc center: {constraints['optic_disc_center']}, radius: {constraints['optic_disc_radius']}")
print(f"Macula center: {constraints['macula_center']}, radius: {constraints['macula_radius']}")
print(f"Superior arcade angle: {np.degrees(constraints['angle_superior']):.1f} deg")
print(f"Inferior arcade angle: {np.degrees(constraints['angle_inferior']):.1f} deg")
print()
print("=== Normalized (for generative model) ===")
for k, v in constraints['normalized'].items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# Overlay constraints on the original image
plot_constraints_overlay(image, constraints, title=f"Constraints: {image_path.name}")

In [ ]:
# Show vessel segmentation and density map side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(constraints['vessel_mask'], cmap='gray')
axes[1].set_title("Vessel Segmentation")
axes[1].axis("off")

im = axes[2].imshow(constraints['vessel_density_map'], cmap='hot',
                     interpolation='bilinear', origin='lower')
axes[2].set_title("Vessel Density Map")
fig.colorbar(im, ax=axes[2], shrink=0.8)

plt.tight_layout()
plt.show()

## 3. Generate Vascular Trees

### 3a. Baseline (manual parameters)

In [ ]:
# Baseline with default/manual parameters
baseline_config = TreeGeneratorConfig(
    retina_radius=1.0,
    root=(0.18, 0.0),
    initial_length=0.23,
    alpha=0.72,
    max_depth=6,
    base_angle_up=np.deg2rad(155),
    base_angle_down=np.deg2rad(205),
    branch_angle_mean=np.deg2rad(28),
    branch_angle_std=np.deg2rad(7),
    macula_center=(-0.25, 0.0),
    macula_radius=0.16,
    random_seed=42,
)

baseline_gen = RetinalTreeGenerator(baseline_config)
baseline_gen.generate()

print(f"Baseline: {len(baseline_gen.edges)} edges, "
      f"{len(baseline_gen.terminal_nodes())} terminals, "
      f"total length = {baseline_gen.total_length():.4f}")

plot_vascular_tree(baseline_gen, title="Baseline Model (manual parameters)")

### 3b. Image-Constrained (extracted parameters)

In [ ]:
# Image-constrained with extracted parameters
constrained_config = TreeGeneratorConfig.from_constraints(
    constraints,
    alpha=0.72,
    max_depth=6,
    branch_angle_mean_deg=28,
    branch_angle_std_deg=7,
    density_weight=0.0,  # No density modulation yet
    random_seed=42,
)

constrained_gen = RetinalTreeGenerator(constrained_config)
constrained_gen.generate()

print(f"Constrained: {len(constrained_gen.edges)} edges, "
      f"{len(constrained_gen.terminal_nodes())} terminals, "
      f"total length = {constrained_gen.total_length():.4f}")

plot_vascular_tree(constrained_gen, title="Image-Constrained Model")

### 3c. Density-Aware Generation

In [ ]:
# Density-aware with extracted parameters + density map
density_config = TreeGeneratorConfig.from_constraints(
    constraints,
    alpha=0.72,
    max_depth=6,
    branch_angle_mean_deg=28,
    branch_angle_std_deg=7,
    density_weight=0.5,
    random_seed=42,
)

density_gen = ConstrainedTreeGenerator(density_config)
density_gen.generate()

print(f"Density-aware: {len(density_gen.edges)} edges, "
      f"{len(density_gen.terminal_nodes())} terminals, "
      f"total length = {density_gen.total_length():.4f}")

plot_vascular_tree(density_gen, title="Density-Aware Constrained Model")

## 4. Side-by-Side Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

plot_vascular_tree(baseline_gen, title="Baseline", ax=axes[0], show=False)
plot_vascular_tree(constrained_gen, title="Image-Constrained", ax=axes[1], show=False)
plot_vascular_tree(density_gen, title="Density-Aware", ax=axes[2], show=False)

fig.suptitle(f"Comparison: {image_path.name}", fontsize=14)
plt.tight_layout()
plt.savefig(str(project_root / "figures" / "comparison_01_h.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5. Quantitative Evaluation

In [ ]:
def evaluate_generator(gen, name, target_density=None):
    """Compute all metrics for a generator."""
    terminals = gen.terminal_nodes()
    terminal_pts = [(n.x, n.y) for n in terminals]
    all_pts = [(n.x, n.y) for n in gen.nodes]
    edge_lengths = [e.length for e in gen.edges]
    
    cov = coverage_score(terminal_pts)
    cov_u = coverage_uniformity(terminal_pts)
    lpt = length_per_terminal(edge_lengths, len(terminals))
    fd, _, _ = fractal_dimension_box_counting(all_pts)
    ba = branch_angle_statistics(gen.edges)
    
    result = {
        'model': name,
        'n_edges': len(gen.edges),
        'n_terminals': len(terminals),
        'total_length': gen.total_length(),
        'coverage_score': cov,
        'coverage_uniformity': cov_u,
        'length_per_terminal': lpt,
        'fractal_dimension': fd,
        'n_bifurcations': ba['n_bifurcations'],
        'mean_branch_angle': ba['mean_total_angle'],
        'branch_angle_std': ba['std_total_angle'],
        'mean_asymmetry': ba['mean_asymmetry'],
    }
    
    if target_density is not None:
        result['density_corr'] = density_correlation(terminal_pts, target_density)
    
    return result


target_density = constraints['vessel_density_map']

results = [
    evaluate_generator(baseline_gen, "Baseline", target_density),
    evaluate_generator(constrained_gen, "Image-Constrained", target_density),
    evaluate_generator(density_gen, "Density-Aware", target_density),
]

df = pd.DataFrame(results).set_index('model')
print(df.round(4).to_string())

## 6. Branch Angle Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, gen, name in zip(axes, 
                          [baseline_gen, constrained_gen, density_gen],
                          ["Baseline", "Image-Constrained", "Density-Aware"]):
    ba = branch_angle_statistics(gen.edges)
    if ba['angles_deg']:
        all_angles = [abs(a) for pair in ba['angles_deg'] for a in pair]
        ax.hist(all_angles, bins=15, edgecolor='black', alpha=0.7, color='steelblue')
        ax.axvline(ba['mean_total_angle']/2, color='red', linestyle='--', 
                   label=f"Mean = {ba['mean_total_angle']/2:.1f} deg")
    ax.set_title(f"{name}\n({ba['n_bifurcations']} bifurcations)")
    ax.set_xlabel("Branch angle (degrees)")
    ax.set_ylabel("Count")
    ax.legend()

plt.suptitle("Branch Angle Distribution", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Batch Processing: All 15 Images

In [ ]:
# Process all healthy images
image_files = sorted(data_dir.glob("*_h.jpg"))
print(f"Found {len(image_files)} images")

all_results = []

for img_path in image_files:
    print(f"Processing {img_path.name}...", end=" ")
    
    img = cv2.imread(str(img_path))
    if img is None:
        print("FAILED to load")
        continue
    
    try:
        cst = extract_all_constraints(img, density_grid_size=20)
        
        # Baseline
        bg = RetinalTreeGenerator(TreeGeneratorConfig(random_seed=42))
        bg.generate()
        
        # Image-constrained
        cc = TreeGeneratorConfig.from_constraints(cst, density_weight=0.0, random_seed=42)
        cg = RetinalTreeGenerator(cc)
        cg.generate()
        
        # Density-aware
        dc = TreeGeneratorConfig.from_constraints(cst, density_weight=0.5, random_seed=42)
        dg = ConstrainedTreeGenerator(dc)
        dg.generate()
        
        target_d = cst['vessel_density_map']
        
        all_results.append({**evaluate_generator(bg, "Baseline", target_d), 'image': img_path.name})
        all_results.append({**evaluate_generator(cg, "Image-Constrained", target_d), 'image': img_path.name})
        all_results.append({**evaluate_generator(dg, "Density-Aware", target_d), 'image': img_path.name})
        
        print("OK")
    except Exception as e:
        print(f"ERROR: {e}")

batch_df = pd.DataFrame(all_results)
print(f"\nTotal results: {len(batch_df)} rows")

In [ ]:
# Summary statistics grouped by model type
summary = batch_df.groupby('model').agg({
    'n_edges': 'mean',
    'n_terminals': 'mean',
    'total_length': 'mean',
    'coverage_score': 'mean',
    'coverage_uniformity': 'mean',
    'fractal_dimension': 'mean',
    'mean_branch_angle': 'mean',
    'density_corr': 'mean',
}).round(4)

print("=== Average Metrics Across All Images ===")
print(summary.to_string())

In [ ]:
# Save results
batch_df.to_csv(str(project_root / "results" / "batch_evaluation.csv"), index=False)
summary.to_csv(str(project_root / "results" / "model_comparison_summary.csv"))
print("Results saved to results/")